# 12 · Pandas: Hands-on Tutorial

We'll take a **messy real-world-style dataset** from an online shop and turn it into business answers.
This is the everyday workflow of a data analyst:

1. **Load** the data
2. **Explore** it (shape, types, missing values)
3. **Clean** it (duplicates, bad text, wrong types, missing and invalid values)
4. **Combine** it with another table
5. **Analyse** it (groupby, pivot tables, trends)
6. **Save** the clean result

Read the notes in [README.md](README.md) alongside this notebook. Every code cell's output is saved, so you can read it on GitHub without running anything.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
print("pandas", pd.__version__)

pandas 3.1.0


## 1. Series and DataFrame: the two building blocks

- A **Series** is one labelled column: values + an **index**.
- A **DataFrame** is a table: several Series that share the same index.

In [2]:
prices = pd.Series([1999, 499, 4999], index=["Earbuds", "Charger", "Watch"], name="price")
prices

Earbuds    1999
Charger     499
Watch      4999
Name: price, dtype: int64

In [3]:
prices["Charger"], prices.mean(), (prices * 1.18).round(0).tolist()

(np.int64(499), np.float64(2499.0), [2359.0, 589.0, 5899.0])

In [4]:
small = pd.DataFrame({
    "product": ["Earbuds", "Charger", "Watch"],
    "price": [1999, 499, 4999],
    "in_stock": [True, True, False],
})
small

,product,price,in_stock
0,Earbuds,1999,True
1,Charger,499,True
2,Watch,4999,False


## 2. Load and explore

`read_csv` turns a CSV file into a DataFrame. The first thing to do with any new dataset is **look at it**.

In [5]:
orders = pd.read_csv("data/orders.csv")
products = pd.read_csv("data/products.csv")
orders.head()

,order_id,order_date,customer,city,product_id,quantity,payment_method
0,1001,2026-06-25,Dev,PUNE,P07,3.0,UPI
1,1002,2026-01-23,Priya,New Delhi,P12,1.0,UPI
2,1003,2026-02-19,Isha,mumbai,P04,1.0,UPI
3,1004,2026-05-12,Ravi,mumbai,P04,NaN,Card
4,1004,2026-05-12,Ravi,mumbai,P04,NaN,Card


In [6]:
orders.shape  # (rows, columns)

(305, 7)

In [7]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 305 entries, 0 to 304
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        305 non-null    int64  
 1   order_date      305 non-null    str    
 2   customer        305 non-null    str    
 3   city            305 non-null    str    
 4   product_id      305 non-null    str    
 5   quantity        292 non-null    float64
 6   payment_method  297 non-null    str    
dtypes: float64(1), int64(1), str(5)
memory usage: 25.7 KB


`info()` already tells us a lot:
- `order_date` is text (`str`), not a date, so we can't do date maths on it yet.
- `quantity` is a float because it has missing values (`NaN` is a float).
- `quantity` and `payment_method` have fewer non-null values than there are rows.

In [8]:
orders.describe()  # statistics for the numeric columns

,order_id,quantity
count,305.000000,292.000000
mean,1149.740984,1.712329
std,86.804085,0.799131
min,1001.000000,-2.000000
25%,1075.000000,1.000000
50%,1150.000000,2.000000
75%,1224.000000,2.000000
max,1300.000000,3.000000


**Spot the problem:** the minimum quantity is **-2**. You can't order a negative number of items, so that's a data-entry error we must handle.

In [9]:
orders.isna().sum()  # missing values per column

order_id           0
order_date         0
customer           0
city               0
product_id         0
quantity          13
payment_method     8
dtype: int64

In [10]:
orders["city"].value_counts()

city
Mumbai       51
mumbai       30
delhi        27
pune         24
Delhi        23
PUNE         22
New Delhi    21
 Pune        21
Hyderabad    18
Bangalore    17
hyderabad    16
Pune         15
bengaluru    12
Bengaluru     8
Name: count, dtype: int64

**14 spellings for 5 cities!** Extra spaces, different capitalisation, and old names (*Bangalore* vs *Bengaluru*, *New Delhi* vs *Delhi*). Any "sales by city" report would be wrong until we fix this.

In [11]:
orders.duplicated().sum()  # rows that are exact copies of an earlier row

np.int64(5)

## 3. Selecting data

| You want | Use |
|----------|-----|
| One column (a Series) | `df["col"]` |
| Several columns | `df[["col1", "col2"]]` |
| Rows and columns **by label** | `df.loc[rows, cols]` |
| Rows and columns **by position** | `df.iloc[rows, cols]` |
| Rows that match a condition | `df[df["col"] > 5]` |

In [12]:
orders[["order_id", "customer", "quantity"]].head(3)

,order_id,customer,quantity
0,1001,Dev,3.0
1,1002,Priya,1.0
2,1003,Isha,1.0


In [13]:
orders.loc[0:2, ["customer", "city"]]   # loc: labels, the end label IS included

,customer,city
0,Dev,PUNE
1,Priya,New Delhi
2,Isha,mumbai


In [14]:
orders.iloc[0:2, 0:3]   # iloc: positions, the end is EXCLUDED (like Python slicing)

,order_id,order_date,customer
0,1001,2026-06-25,Dev
1,1002,2026-01-23,Priya


In [15]:
# Boolean filtering, the same idea as NumPy masks. Use & | ~ and brackets.
orders[(orders["customer"] == "Asha") & (orders["quantity"] >= 2)]

,order_id,order_date,customer,city,product_id,quantity,payment_method
17,1017,2026-04-01,Asha,Mumbai,P02,2.0,Card
38,1038,2026-04-15,Asha,Mumbai,P09,3.0,UPI
83,1082,2026-06-13,Asha,New Delhi,P07,2.0,UPI
94,1093,2026-05-29,Asha,hyderabad,P09,3.0,Card
152,1150,2026-01-18,Asha,PUNE,P11,3.0,Cash on Delivery
173,1170,2026-01-15,Asha,mumbai,P07,2.0,UPI
188,1185,2026-06-26,Asha,pune,P01,2.0,Card
203,1200,2026-02-02,Asha,mumbai,P01,3.0,Cash on Delivery
253,1249,2026-05-22,Asha,PUNE,P04,2.0,Cash on Delivery


In [16]:
orders.query("customer == 'Asha' and quantity >= 2")  # same thing, sometimes easier to read

,order_id,order_date,customer,city,product_id,quantity,payment_method
17,1017,2026-04-01,Asha,Mumbai,P02,2.0,Card
38,1038,2026-04-15,Asha,Mumbai,P09,3.0,UPI
83,1082,2026-06-13,Asha,New Delhi,P07,2.0,UPI
94,1093,2026-05-29,Asha,hyderabad,P09,3.0,Card
152,1150,2026-01-18,Asha,PUNE,P11,3.0,Cash on Delivery
173,1170,2026-01-15,Asha,mumbai,P07,2.0,UPI
188,1185,2026-06-26,Asha,pune,P01,2.0,Card
203,1200,2026-02-02,Asha,mumbai,P01,3.0,Cash on Delivery
253,1249,2026-05-22,Asha,PUNE,P04,2.0,Cash on Delivery


## 4. Cleaning

We work on a copy and keep the raw data untouched, so we can always compare and start again.

### 4.1 Remove duplicates

In [17]:
clean = orders.drop_duplicates().copy()
print(len(orders), "->", len(clean), "rows")

305 -> 300 rows


### 4.2 Fix the text in `city`

The `.str` accessor applies string methods to a whole column at once (vectorised, like NumPy).

In [18]:
clean["city"] = (
    clean["city"]
    .str.strip()      # remove spaces at both ends
    .str.title()      # "PUNE" -> "Pune"
    .replace({"Bangalore": "Bengaluru", "New Delhi": "Delhi"})
)
clean["city"].value_counts()

city
Pune         81
Mumbai       80
Delhi        69
Bengaluru    37
Hyderabad    33
Name: count, dtype: int64

Five clean cities. ✅

### 4.3 Convert types

In [19]:
clean["order_date"] = pd.to_datetime(clean["order_date"])
clean["order_date"].dtype

dtype('<M8[us]')

### 4.4 Invalid and missing values

Each choice here is a **decision you should be able to explain**:
- A quantity of 0 or less is impossible, so we treat it as **missing** too.
- For missing quantities we use the **most common value** (the mode). Most orders are for 1 item, so that's the safest guess, and only about 5% of rows are affected.
- For a missing payment method we can't guess, so we label it `"Unknown"` instead of deleting the order.

In [20]:
invalid = clean["quantity"] <= 0
print("invalid quantities:", invalid.sum())
clean.loc[invalid, "quantity"] = np.nan

most_common = int(clean["quantity"].mode()[0])
print("missing quantities:", clean["quantity"].isna().sum(), "-> filled with", most_common)
clean["quantity"] = clean["quantity"].fillna(most_common).astype(int)

clean["payment_method"] = clean["payment_method"].fillna("Unknown")
clean.isna().sum()

invalid quantities: 2
missing quantities: 14 -> filled with 1


order_id          0
order_date        0
customer          0
city              0
product_id        0
quantity          0
payment_method    0
dtype: int64

No missing values left, and `quantity` is a proper integer column now.

> **pandas 3 note:** always change data with `df.loc[rows, "col"] = value` or `df["col"] = ...`.
> "Chained" assignment like `df["col"][rows] = value` never changes the DataFrame in pandas 3 (Copy-on-Write).

## 5. Combining tables: `merge`

Orders only have a `product_id`. The product name, category and price live in `products.csv`. `merge` joins them, like a SQL JOIN or Excel's VLOOKUP.

In [21]:
products

,product_id,product,category,price
0,P01,Wireless Earbuds,Electronics,1999
1,P02,Phone Charger,Electronics,499
2,P03,Smart Watch,Electronics,4999
3,P04,Python Crash Course,Books,899
4,P05,Data Science Handbook,Books,1299
5,P06,Notebook Pack,Books,249
6,P07,Coffee Mug,Home,349
7,P08,Desk Lamp,Home,1199
8,P09,Cushion Cover,Home,399
9,P10,Cotton Kurta,Clothing,1499


In [22]:
merged = clean.merge(products, on="product_id", how="left", indicator=True)
merged["_merge"].value_counts()

_merge
both          297
left_only       3
right_only      0
Name: count, dtype: int64

`how="left"` keeps **every order**, even when the product isn't found. `indicator=True` adds a `_merge` column that shows which rows matched. Three orders point to `P99`, a product that doesn't exist.

In [23]:
merged[merged["_merge"] == "left_only"][["order_id", "product_id", "product", "price"]]

,order_id,product_id,product,price
55,1056,P99,NaN,NaN
201,1202,P99,NaN,NaN
260,1261,P99,NaN,NaN


In [24]:
# An inner join would silently drop them. Dropping them is the right call here,
# but now we know exactly what we dropped, and why.
sales = merged[merged["_merge"] == "both"].drop(columns="_merge")
len(sales)

297

**A hidden side effect:** the 3 unmatched rows had `NaN` as their price, and `NaN` is a float, so the whole `price` column became float (`1999.0`). Now that those rows are gone, we convert it back to whole numbers.

In [25]:
print("before:", sales["price"].dtype)
sales["price"] = sales["price"].astype(int)
print("after: ", sales["price"].dtype)

before: float64
after:  int64


## 6. New columns

Prefer **vectorised** column maths over `apply`. It's faster and clearer.

In [26]:
sales["revenue"] = sales["quantity"] * sales["price"]
sales["month"] = sales["order_date"].dt.to_period("M")
sales["weekday"] = sales["order_date"].dt.day_name()
sales["order_size"] = pd.cut(
    sales["revenue"], bins=[0, 500, 2000, np.inf], labels=["Small", "Medium", "Large"]
)
sales["is_big_order"] = np.where(sales["revenue"] >= 3000, "Yes", "No")
sales[["order_id", "product", "quantity", "price", "revenue", "month", "order_size"]].head()

,order_id,product,quantity,price,revenue,month,order_size
0,1001,Coffee Mug,3,349,1047,2026-06,Medium
1,1002,Denim Jacket,1,2499,2499,2026-01,Large
2,1003,Python Crash Course,1,899,899,2026-02,Medium
3,1004,Python Crash Course,1,899,899,2026-05,Medium
4,1005,Wireless Earbuds,2,1999,3998,2026-03,Large


## 7. Analysis: answering business questions

### Q1. What are the headline numbers?

In [27]:
print(f"Orders:            {len(sales)}")
print(f"Revenue:           ₹{sales['revenue'].sum():,}")
print(f"Average order:     ₹{sales['revenue'].mean():,.0f}")
print(f"Customers:         {sales['customer'].nunique()}")

Orders:            297
Revenue:           ₹685,496
Average order:     ₹2,308
Customers:         12


### Q2. Which category earns the most?

`groupby` = **split** the rows into groups, **apply** a calculation to each group, **combine** the results.

In [28]:
by_category = (
    sales.groupby("category")
    .agg(orders=("order_id", "count"),
         items=("quantity", "sum"),
         revenue=("revenue", "sum"))
    .sort_values("revenue", ascending=False)
)
by_category["share_%"] = (by_category["revenue"] / by_category["revenue"].sum() * 100).round(1)
by_category

,orders,items,revenue,share_%
category,,,,
Electronics,94,172,295828,43.2
Clothing,59,102,226898,33.1
Books,73,122,96528,14.1
Home,71,108,66242,9.7


### Q3. What are the top 5 products by revenue?

In [29]:
sales.groupby("product")["revenue"].sum().nlargest(5)

product
Wireless Earbuds    141929
Smart Watch         114977
Running Shoes        89970
Denim Jacket         72471
Cotton Kurta         64457
Name: revenue, dtype: int64

### Q4. Is the business growing month by month?

In [30]:
monthly = sales.groupby("month").agg(orders=("order_id", "count"), revenue=("revenue", "sum"))
monthly["growth_%"] = (monthly["revenue"].pct_change() * 100).round(1)
monthly

,orders,revenue,growth_%
month,,,
2026-01,30,68806,NaN
2026-02,32,91189,32.5
2026-03,40,74043,-18.8
2026-04,52,126411,70.7
2026-05,64,142488,12.7
2026-06,79,182559,28.1


### Q5. Which city buys what? (a pivot table)

In [31]:
pd.pivot_table(sales, values="revenue", index="city", columns="category",
               aggfunc="sum", fill_value=0, margins=True, margins_name="Total")

category,Books,Clothing,Electronics,Home,Total
city,,,,,
Bengaluru,7038,19489,21980,7733,56240
Delhi,27516,57475,37975,13876,136842
Hyderabad,10590,27487,49981,7788,95846
Mumbai,31511,28986,105941,14624,181062
Pune,19873,93461,79951,22221,215506
Total,96528,226898,295828,66242,685496


### Q6. How do customers pay?

In [32]:
sales["payment_method"].value_counts(normalize=True).mul(100).round(1)

payment_method
UPI                 51.5
Card                25.3
Cash on Delivery    20.5
Unknown              2.7
Name: proportion, dtype: float64

### Q7. Who are the best customers?

In [33]:
(sales.groupby("customer")
      .agg(orders=("order_id", "count"), revenue=("revenue", "sum"))
      .sort_values("revenue", ascending=False)
      .head(3))

,orders,revenue
customer,,
Vikram,36,79687
Isha,23,72807
Sana,28,63955


### Q8. How big are orders?

In [34]:
pd.crosstab(sales["category"], sales["order_size"])

order_size,Small,Medium,Large
category,,,
Books,19,43,11
Clothing,0,8,51
Electronics,19,36,39
Home,26,38,7


## 8. Save the clean data

The next topic (visualisation) will start from this clean file, so cleaning happens only once.

In [35]:
sales.to_csv("data/sales_clean.csv", index=False)
print("saved", sales.shape)

saved (297, 15)


## 9. Bonus: your Expense Tracker data in pandas

The mini project in topic 10 produces exactly the kind of data pandas is made for. Here we load its demo data directly from the project's own code.

In [36]:
import sys
sys.path.insert(0, "../10_mini_project")
from sample_data import demo_tracker

expenses = pd.DataFrame([e.to_dict() for e in demo_tracker()])
expenses["date"] = pd.to_datetime(expenses["date"])
expenses.groupby("category")["amount"].agg(["count", "sum"]).sort_values("sum", ascending=False)

,count,sum
category,,
bills,3,9539.0
food,4,4860.0
shopping,2,2301.0
transport,2,1280.0
fun,1,600.0
health,1,360.0


The same numbers as the tracker's summary screen, in one line of pandas.

## What you learned

| Step | Tools |
|------|-------|
| Load | `read_csv` |
| Explore | `head`, `info`, `describe`, `isna().sum()`, `value_counts`, `duplicated` |
| Select | `[]`, `loc`, `iloc`, boolean masks, `query` |
| Clean | `drop_duplicates`, `.str`, `replace`, `to_datetime`, `fillna`, `astype` |
| Combine | `merge` (with `indicator=True`), `concat` |
| Create | vectorised maths, `.dt`, `pd.cut`, `np.where` |
| Analyse | `groupby().agg()`, `nlargest`, `pct_change`, `pivot_table`, `crosstab` |
| Save | `to_csv` |

**Next:** topic 13 turns these tables into charts with Matplotlib and Seaborn.